# Step 6
s3://thesis--ec331-s3/capped-volume-bids/

In [ ]:
import boto3
import pandas as pd

# Initialize S3 client
s3 = boto3.client('s3')
bucket_name = "thesis--ec331-s3"
prefix = "melted-volume-bids/"

# List all parquet files in the directory
print("Listing parquet files...")
response = s3.list_objects_v2(Bucket=bucket_name, Prefix=prefix)
parquet_files = [obj['Key'] for obj in response.get('Contents', []) 
                if obj['Key'].endswith('.parquet')]

# Handle pagination if there are many files
while response.get('IsTruncated', False):
    response = s3.list_objects_v2(
        Bucket=bucket_name, 
        Prefix=prefix,
        ContinuationToken=response['NextContinuationToken']
    )
    parquet_files.extend([obj['Key'] for obj in response.get('Contents', []) 
                        if obj['Key'].endswith('.parquet')])

print(f"Found {len(parquet_files)} parquet files")

# Load all files
print("Loading files into DataFrame...")
dfs = []
for file_key in parquet_files:
    try:
        s3_uri = f"s3://{bucket_name}/{file_key}"
        df = pd.read_parquet(s3_uri)
        dfs.append(df)
    except Exception as e:
        print(f"Error reading {file_key}: {e}")

# Concatenate all dataframes
print("Concatenating all DataFrames...")
if dfs:
    combined_df = pd.concat(dfs, ignore_index=True)
    
    # Get initial info
    total_rows = len(combined_df)
    print(f"Total rows in combined DataFrame: {total_rows}")
    print(f"DataFrame shape: {combined_df.shape}")
    print(f"Memory usage: {combined_df.memory_usage(deep=True).sum() / 1e9:.2f} GB")
    
    # Check for duplicates
    print("Checking for duplicates...")
    duplicated = combined_df.duplicated()
    duplicate_count = duplicated.sum()
    duplicate_percentage = (duplicate_count / total_rows) * 100
    
    print(f"Found {duplicate_count} duplicate rows ({duplicate_percentage:.2f}%)")
    
    # Show examples of duplicated rows if there are any
    if duplicate_count > 0:
        print("\nExample of duplicated rows:")
        duplicate_examples = combined_df[duplicated].head(3)
        print(duplicate_examples)
    
    # Get deduplicated DataFrame if needed
    print("\nRemoving duplicates...")
    deduplicated_df = combined_df.drop_duplicates()
    print(f"DataFrame shape after deduplication: {deduplicated_df.shape}")
    
    # Optional: save deduplicated data
    # deduplicated_df.to_parquet("s3://thesis--ec331-s3/deduplicated-data.parquet")
else:
    print("No data found in the parquet files")

Listing parquet files...
Found 2549 parquet files
Loading files into DataFrame...
